# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q networkx matplotlib pandas faker
    print('Packages installed.')


# Bipartite Network Analysis (Affiliation Networks)

Most social networks we study are "One-Mode" (Person-Person). However, data often comes in "Two-Mode" or **Bipartite** forms: People attending Events, Directors on Boards, or Authors on Papers.

This notebook demonstrates how to handle these networks using `networkx`, specifically projecting them into one-mode networks to study the relationships implied by shared affiliations.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import random
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

## 1. Generating Bipartite Data
We create two sets of nodes: `People` and `Clubs`.

In [ ]:
people = [fake.first_name() for _ in range(10)]
clubs = ['Chess Club', 'Debate Team', 'Gardening', 'Book Club', 'Running Group']

B = nx.Graph()
B.add_nodes_from(people, bipartite=0) # Add people (set 0)
B.add_nodes_from(clubs, bipartite=1) # Add clubs (set 1)

# Randomly assign people to clubs
for person in people:
    # Each person joins 1-3 clubs
    n_clubs = random.randint(1, 3)
    joined_clubs = random.sample(clubs, n_clubs)
    for club in joined_clubs:
        B.add_edge(person, club)

print(f"Nodes: {B.number_of_nodes()}")
print(f"Edges: {B.number_of_edges()}")

## 2. Visualizing the Bipartite Graph
We use a bipartite layout to clearly separate the two sets.

In [ ]:
pos = nx.bipartite_layout(B, people)

plt.figure(figsize=(10, 6))
nx.draw_networkx_nodes(B, pos, nodelist=people, node_color='skyblue', label='People')
nx.draw_networkx_nodes(B, pos, nodelist=clubs, node_color='salmon', label='Clubs', node_shape='s')
nx.draw_networkx_edges(B, pos, alpha=0.5)
nx.draw_networkx_labels(B, pos)

plt.legend()
plt.title("Bipartite Network: People <-> Clubs")
plt.axis('off')
plt.show()

## 3. One-Mode Projection
We often want to know: *Who knows whom?* We assume that if two people attend the same club, they have a connection. This is a "Projected Graph".

In [ ]:
# Project onto People nodes
P = nx.bipartite.weighted_projected_graph(B, people)

print("Edges in Projected Person-Person Network:")
for u, v, data in P.edges(data=True):
    print(f"{u} <-> {v} (Shared Clubs: {data['weight']})")

# Viz Projection
plt.figure(figsize=(8, 8))
pos_p = nx.spring_layout(P, seed=42)

# Edge width based on weight (shared clubs)
weights = [data['weight'] * 2 for u, v, data in P.edges(data=True)]

nx.draw(P, pos_p, with_labels=True, node_color='skyblue', node_size=1500, width=weights, edge_color='gray')
plt.title("Projected Person-Person Network (Weighted)")
plt.show()

## 4. Centrality in Projected Network
Who is the most central person based on shared affiliations?

In [ ]:
degree_centrality = nx.degree_centrality(P)
sorted_degree = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)

print("Degree Centrality (Projected):")
for person, cent in sorted_degree:
    print(f"{person}: {cent:.2f}")